In [39]:
import os
import PIL 
import json
import cv2
from pycocotools.coco import COCO

In [41]:
img_dir = r"D:\2025\data\28163228\petals-anonymous"
ann_file = r"D:\2025\data\28163228\petals-anonymous.json"

In [55]:
imgs_path = {
   os.path.basename(i) : os.path.join(img_dir, i) for i in os.listdir(img_dir)
}

In [22]:
def help_cocoann2segann(imgfiles, annfile, savefile, class_id):
    anns = COCO(annfile)
    imgs = {i: cv2.imread(i) for i in imgfiles}
    ann = [i for i in ann if i['category_id'] == class_id]
    img = [i for i in img if i['id'] == ann[0]['image_id']]

In [7]:
ann = json.load(open(ann_file, 'r'))

In [45]:
ann['images'].__len__()

1026

In [7]:
import torch
import numpy as np
import onnxruntime
import traceback
import cv2

def to_numpy(tensor):
    return tensor.detach().cpu().numpy() if tensor.requires_grad else tensor.cpu().numpy()

# 模型文件路径
onnx_model_path = r"D:\2025\weights_from_online\model2_segmentation.onnx"

try:
    # 加载ONNX模型
    sess = onnxruntime.InferenceSession(onnx_model_path)

    # 获取模型输入名称 (实际名称，需要用Netron确认！)
    # input_name = sess.get_inputs()[0].name
    # **** 重要：用 Netron 查看你的 ONNX 文件，确认第一个（或唯一的）图像输入节点的真实名称！ ****
    # **** 它很可能不是 "input_dict_list"，可能是 "image", "input", "input.1" 等 ****
    input_name = "input_dict_list" # <---- 假设真实名称是 "image"，请根据 Netron 修改！
    print(f"使用模型输入名称: {input_name}")

    # 创建测试图像数据 (NumPy array)
    # 形状应与模型期望匹配 (例如 NCHW: [1, 3, 512, 1024] 或 CHW: [3, 512, 1024])
    # **** 重要：根据你最终确认的模型期望形状来创建！ ****
    # 假设模型期望 NCHW [1, 3, H, W] (Rank 4)
    dummy_image_np = cv2.imread(r"D:\2025\data\28163228\samsung-a52-trays-anonymous\2a29bf59-d2df-47ba-939d-258e5661fbee.png")
    print(f"原始图像形状: {dummy_image_np.shape}")
    # 调整图像大小以匹配模型期望 (例如 512x1024)
    dummy_image_np = cv2.resize(dummy_image_np, (512, 512)).transpose(2, 1, 0).astype(np.float32)
    print(f"调整后图像形状: {dummy_image_np.shape}")
    # 或者，如果模型期望 CHW [3, H, W] (Rank 3)
    # dummy_image_np = np.random.randn(3, 512, 1024).astype(np.float32)

    # 准备 input_feed 字典
    input_feed = {'input_image': dummy_image_np.reshape(1, *dummy_image_np.shape)}

    # 获取输出节点名称列表（可选，None 会返回所有输出）
    output_names = [output.name for output in sess.get_outputs()]
    print(f"模型输出节点: {output_names}")

    # 执行ONNX模型推理
    print("开始推理...")
    onnx_out = sess.run(output_names, input_feed)
    print("推理完成。")

    # 打印输出结果及形状
    print("模型输出结果数量:", len(onnx_out))
    for i, output in enumerate(onnx_out):
         # 检查输出是否是 NumPy 数组 (通常是)
         if isinstance(output, np.ndarray):
             print(f"输出 '{output_names[i]}' (类型 {output.dtype}) 的形状: {output.shape}")
         else:
             print(f"输出 '{output_names[i]}' 的类型: {type(output)}, 值: {output}") # 处理非 ndarray 输出

except Exception as e:
    print(f"模型加载或推理过程出错: {str(e)}")
    traceback.print_exc()


使用模型输入名称: input_dict_list
原始图像形状: (4624, 3468, 3)
调整后图像形状: (3, 512, 512)
模型输出节点: ['output_segmentation_map']
开始推理...
推理完成。
模型输出结果数量: 1
输出 'output_segmentation_map' (类型 float32) 的形状: (1, 1, 512, 512)


In [1]:
from utils.inference_onnx import InferenceRoseDisease
infer = InferenceRoseDisease(model1_onnx = r"model.onnx", 
                            model2_onnx = r"D:\2025\weights_from_online\model2_segmentation.onnx",     
                            model3_onnx = r"D:\2025\weights_from_online\model3_segmentation.onnx", 
                            )

c:\Users\16477\miniconda3\envs\det2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
# infer.step1_crop_img(r"D:\2025\data\obj_dectetion\data_path_o\DSC01011_1.JPG",
#                     'demo/demo2')

from utils.for_petal_data import split_dataset
split_dataset(r"D:\2025\data\obj_for_per_sam\seg_petal\train",
              r"D:\2025\data\obj_for_per_sam\seg_petal\val")

c:\Users\16477\miniconda3\envs\det2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
infer.step2_seg_petal('demo\demo2\DSC01011_1_1.jpg', save_dir='demo\demo2\petal')